# STEP 0 — Understand What ch_overview Is

This table contains:
- company master data
- legal information
- incorporation details
- accounts information
- company status
- SIC codes
- registered office info

This table becomes our MASTER COMPANY TABLE in Silver.
Bronze vs Silver Thinking
Bronze - Raw API respons which Keep:
- nested JSON
- raw column names
- duplicates
- source fidelity

Silver - Business-cleaned data
We:
- flatten nested objects
- clean column names
- convert dates
- remove duplicates
- standardize schema
- prepare for analytics


# STEP 1 — Import Required Libraries and Inspect Data/Schema Print

### 1 - pyspark.sql.functions Contains transformation functions like:
- trim()
- col()
- to_date()
- explode()

### 2 - DeltaTable Used for:
- MERGE
- UPSERT
- SCD Type 1

### 3- create sparkSession Explanation
This gets the active Spark session running in Databricks.

We need this to:
- read tables
- write tables
- run transformations

### 4 — Read Bronze Table
spark.read.table
Explanation - This loads the Bronze table into a Spark DataFrame.
Think of DataFrame as distributed Excel table but scalable.

### 5 - Inspect Data

Before transforming ANYTHING understand the data.

5.1 View Sample Rows
display(df.limit(5))
Why?
To visually inspect:
- nested columns
- nulls
- formatting issues
- strange values

5.2 Print Schema
df.printSchema()
Why?
This is one of the MOST IMPORTANT steps.

We discover:
- string
- integer
- boolean
- struct
- array

Example:
last_accounts: struct - means nested object.


In [0]:
from pyspark.sql import SparkSession ##SparkSession Used to interact with Spark and it is the main entry point.
from pyspark.sql.functions import * 
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

df = spark.read.table(
    "company_risk_intelligence_platform.bronze.ch_overview"
)
display(df.limit(5))

In [0]:
df.printSchema()

# Step 2 — Decide What To Flatten

### Important distinction:
- Structure	Operation
- STRUCT / OBJECT	flatten
- ARRAY	explode

Our accounts column is a STRUCT So we do:
col("accounts.next_due") NOT explode.

### Design the Silver Schema

Here is the proper Silver schema for ch_overview.

- Core Company Info
- company_number
- company_name
- company_status
- company_type
- jurisdiction
- date_of_creation
- can_file
- Accounts Info (Flattened)

From accounts

- accounting_ref_day
- accounting_ref_month
- 
- last_accounts_made_up_to
- last_accounts_period_start
- last_accounts_period_end
- last_accounts_type
- 
- next_accounts_due_on
- next_accounts_period_start
- next_accounts_period_end
- next_accounts_overdue
- 
- next_due
- next_made_up_to
- accounts_overdue
Confirmation Statement
- confirmation_last_made_up_to
- confirmation_next_due
- confirmation_next_made_up_to
- confirmation_overdue
Address Fields Flatten:
- address_line_1
- address_line_2
- address_country
- address_locality
- address_postal_code
- address_region
Business Classification
sic_codes - Keep as ARRAY for now - Do NOT explode in Silver yet.

Governance Flags
- has_been_liquidated
- has_charges
- has_insolvency_history
- has_super_secure_pscs
- registered_office_is_in_dispute
- undeliverable_registered_office_address

Technical Metadata
- last_update_ts
- ingestion_ts
- source_file

#2.1 Transformation Function


In [0]:
# Transformation Function
def transform_ch_overview(df):

    transformed_df = (
        df

        # -----------------------------
        # Core Company Information
        # -----------------------------
        .withColumn("company_type", col("type"))

        # -----------------------------
        # Accounts Information
        # -----------------------------
        .withColumn(
            "accounting_ref_day",
            col("accounts.accounting_reference_date.day")
        )

        .withColumn(
            "accounting_ref_month",
            col("accounts.accounting_reference_date.month")
        )

        .withColumn(
            "last_accounts_made_up_to",
            col("accounts.last_accounts.made_up_to")
        )

        .withColumn(
            "last_accounts_period_start",
            col("accounts.last_accounts.period_start_on")
        )

        .withColumn(
            "last_accounts_period_end",
            col("accounts.last_accounts.period_end_on")
        )

        .withColumn(
            "last_accounts_type",
            col("accounts.last_accounts.type")
        )

        .withColumn(
            "next_accounts_due_on",
            col("accounts.next_accounts.due_on")
        )

        .withColumn(
            "next_accounts_period_start",
            col("accounts.next_accounts.period_start_on")
        )

        .withColumn(
            "next_accounts_period_end",
            col("accounts.next_accounts.period_end_on")
        )

        .withColumn(
            "next_accounts_overdue",
            col("accounts.next_accounts.overdue")
        )

        .withColumn(
            "accounts_next_due",
            col("accounts.next_due")
        )

        .withColumn(
            "accounts_next_made_up_to",
            col("accounts.next_made_up_to")
        )

        .withColumn(
            "accounts_overdue",
            col("accounts.overdue")
        )

        # -----------------------------
        # Confirmation Statement
        # -----------------------------
        .withColumn(
            "confirmation_last_made_up_to",
            col("confirmation_statement.last_made_up_to")
        )

        .withColumn(
            "confirmation_next_due",
            col("confirmation_statement.next_due")
        )

        .withColumn(
            "confirmation_next_made_up_to",
            col("confirmation_statement.next_made_up_to")
        )

        .withColumn(
            "confirmation_overdue",
            col("confirmation_statement.overdue")
        )

        # -----------------------------
        # Registered Office Address
        # -----------------------------
        .withColumn(
            "address_line_1",
            col("registered_office_address.address_line_1")
        )

        .withColumn(
            "address_line_2",
            col("registered_office_address.address_line_2")
        )

        .withColumn(
            "address_country",
            col("registered_office_address.country")
        )

        .withColumn(
            "address_locality",
            col("registered_office_address.locality")
        )

        .withColumn(
            "address_postal_code",
            col("registered_office_address.postal_code")
        )

        .withColumn(
            "address_region",
            col("registered_office_address.region")
        )

        # -----------------------------
        # Rename Metadata Columns
        # -----------------------------
        .withColumnRenamed("file_path", "source_file")

        # -----------------------------
        # Remove Duplicate Companies
        # -----------------------------
        .dropDuplicates(["company_number"])
    )

    # ---------------------------------
    # Final Column Selection
    # ---------------------------------

    transformed_df = transformed_df.select(

        # Core Company Info
        "company_number",
        "company_name",
        "company_status",
        "company_type",
        "jurisdiction",
        "date_of_creation",
        "can_file",

        # Accounts
        "accounting_ref_day",
        "accounting_ref_month",
        "last_accounts_made_up_to",
        "last_accounts_period_start",
        "last_accounts_period_end",
        "last_accounts_type",
        "next_accounts_due_on",
        "next_accounts_period_start",
        "next_accounts_period_end",
        "next_accounts_overdue",
        "accounts_next_due",
        "accounts_next_made_up_to",
        "accounts_overdue",

        # Confirmation Statement
        "confirmation_last_made_up_to",
        "confirmation_next_due",
        "confirmation_next_made_up_to",
        "confirmation_overdue",

        # Address
        "address_line_1",
        "address_line_2",
        "address_country",
        "address_locality",
        "address_postal_code",
        "address_region",

        # Business Classification
        "sic_codes",

        # Governance Flags
        "has_been_liquidated",
        "has_charges",
        "has_insolvency_history",
        "has_super_secure_pscs",
        "registered_office_is_in_dispute",
        "undeliverable_registered_office_address",

        # Metadata
        "last_update_ts",
        "source_file"
    )

    return transformed_df

### Q1 Why flatten nested objects?
Because Gold analytics becomes MUCH easier.
Instead of: accounts.next_accounts.due_on
we can directly use:next_accounts_due_on

### Q2— Why We Did NOT Explode sic_codes
Current value: ["20590","24410","29320","71121"]

One company can have multiple SIC codes.
If we explode:

company	sic
A	20590
A	24410

- This duplicates company rows.
- That is NOT ideal in Silver.
- Keep array in Silver.
- Explode later in Gold if needed for industry analytics.

# 2.2 — SCD Merge [Logic](url)

In [0]:
def scd_merge_table(spark, source_df, target_table, business_key):

    if not spark.catalog.tableExists(target_table):

        print(f"Creating table: {target_table}")

        source_df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

    else:

        delta_table = DeltaTable.forName(spark, target_table)

        merge_condition = " AND ".join(
            [f"target.{col} = source.{col}" for col in business_key]
        )

        (
            delta_table.alias("target")
            .merge(
                source_df.alias("source"),
                merge_condition
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("Merge Completed")

### 2.3 Main Pipeline

In [0]:
source_table = "company_risk_intelligence_platform.bronze.ch_overview"

target_table = "company_risk_intelligence_platform.silver.ch_overview"

business_key = ["company_number"]

# Read Bronze
df = spark.read.table(source_table)

# Transform
silver_df = transform_ch_overview(df)

# Merge into Silver
scd_merge_table(
    spark,
    silver_df,
    target_table,
    business_key
)

In [0]:
df = spark.read.table(target_table)
display(df)